# v11: v10 + robust line tracing, planned 90° corners and angle-independent green markers

Built on your calibrated v10. Every new change is tagged `# v11:`; v10 changes keep their `# v10:` tags.

| Problem seen on the course | Root cause | v11 change | Embedded-design principle |
|---|---|---|---|
| Two 90° bends close together: robot stops turning early and follows the wrong tape, or cuts the zig-zag | Black-line TURN started while the robot was still on the incoming tape (error ≥ 0.55 from look-ahead) and ended as soon as *any* tape crossed the image centre. Your log: TURN states lasted 0.20–0.46 s, but a 90° pivot takes 0.67 s at your calibration → turns ended at 30–60°. Also, the inner wheel was commanded at −0.09, below the measured stall limit 0.12. | Perception emits an explicit **corner signal**. BlackLineController runs a planned corner: approach → drive the axle over the corner (blind time) → rotate ≥ 63° open-loop → finish only when near, middle *and* far bands show the tape straight ahead. If the next corner is already close, it chains straight into it. Inner-wheel ratio −0.8. | Plan from an explicit event, not a threshold crossing; feedforward + feedback; respect actuator limits |
| Green marker read as CENTER when approaching at an angle | v10 required a *horizontal* crossbar row; at an angle the crossbar is slanted. The slanted crossbar also crossed the near band, corrupting the line position and even flipping LEFT/RIGHT. | **Row-by-row line tracer** from the image bottom: rows much wider than the measured tape width are crossbars/corners and are excluded from the position. Marker validity checked **along the line direction** (crossbar just ahead, nothing behind). | Validate sensor data relative to a measured reference; make perception invariant to pose |

| Next corner only a few cm after a turn | Turn finish needed near **and middle** bands; with a short connector the middle band is empty, so the robot kept rotating. | Finish uses the tracer's measured tape **angle** and the lowest traced rows. If the next corner is already at the image bottom (connector shorter than the camera can see), finish on calibrated rotation time and chain into the next corner. Simulated down to 4 cm. | Graceful degradation: fall back to the model when the sensor cannot see |

Before floor tests: run `self_test()` (17 cases incl. diagonal approaches), then preview at a zig-zag and watch the `corner` column in the log.

## 1. Hardware and camera

In [1]:
import time, csv
from collections import deque, namedtuple
import cv2
import numpy as np
import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()
camera = Camera.instance(width=224, height=224)
live_view = widgets.Image(format="jpeg", width=336, height=336)
camera_link = traitlets.dlink((camera, "value"), (live_view, "value"), transform=bgr8_to_jpeg)
display(live_view)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [2]:
latest = {"sample": None}  # newest frame only: decisions are made on fresh data, never on a queue

def receive_frame(change):
    latest["sample"] = (change["new"], time.monotonic())

camera.observe(receive_frame, names="value")
time.sleep(0.5)
print("Camera frame received." if latest["sample"] is not None else "No camera frame yet.")

Camera frame received.


## 2. Calibration constants (measure these on the real floor)
v10 collects every value that depends on the physical robot in one place, and derives timeouts from
measured quantities instead of tuning each timeout separately.

In [ ]:
import time, csv
from collections import deque, namedtuple
import cv2
import numpy as np

# Motor commands (copy your proven values)
BASE_SPEED = 0.15
CURVE_SPEED = 0.13
MIN_DRIVE_SPEED = 0.12    # v10 CALIBRATE: lowest command that still reliably moves the robot (avoid stalling)
KP = 0.14
MAX_SPEED = 0.25
TURN_SPEED = 0.15         # black-line pivot, outer wheel
GREEN_TURN_SPEED = 0.15   # green turn, both wheels (one reversed)
PIVOT_INNER_RATIO = -0.8  # v11: inner = -0.8 x 0.15 = -0.12 = MIN_DRIVE_SPEED. At -0.6 (-0.09) the inner wheel likely stalled,
                          # so the robot pivoted about that wheel and the camera swung off the corner

# Measured geometry/timing
ROTATION_TIME_360 = 2.15   # v10 CALIBRATE with spin_test(): seconds per full in-place turn at GREEN_TURN_SPEED
CAMERA_BLIND_TIME = 0.12  # v10 CALIBRATE: time at BASE_SPEED for tape leaving the image bottom to reach the axle
FRAME_TIMEOUT = 0.5

# Derived (re-run this cell after changing a calibration value)
QUARTER_TURN_TIME = ROTATION_TIME_360 / 4.0
# Black-line pivot rotates slower: wheel-speed difference is (1 - ratio)*TURN_SPEED instead of 2*GREEN_TURN_SPEED
LINE_PIVOT_TIME_360 = ROTATION_TIME_360 * (2.0 * GREEN_TURN_SPEED) / ((1.0 - PIVOT_INNER_RATIO) * TURN_SPEED)

def clamp(value, low, high):
    return max(low, min(high, value))

In [9]:
def spin_test(seconds=2.15):
    """Spin in place for a fixed time; measure the angle turned and set ROTATION_TIME_360 = seconds*360/angle."""
    try:
        robot.set_motors(-GREEN_TURN_SPEED, GREEN_TURN_SPEED)
        time.sleep(seconds)
    finally:
        robot.stop()

spin_test(2.15)

## 3. Perception
v11: the line is found by a **row-by-row tracer** that starts at the image bottom and skips wide rows (crossbars, corners), so the line position is correct at any approach angle. Wide rows become a branch / corner signal. Marker validity is checked along the line direction instead of with a horizontal row.

In [ ]:
BLACK_THRESHOLD = 70
BANDS = [("near", 0.72, 0.88), ("middle", 0.54, 0.70), ("far", 0.36, 0.52)]

# v11: row-by-row line tracer (replaces "largest contour centroid per band")
TRACE_BOTTOM = 0.95         # start just above the image bottom, where the robot's own tape always is
TRACE_TOP = 0.30
TRACE_MAX_RUN = 0.20        # a black run wider than this fraction of the width is a crossbar / branch row
TRACE_MAX_SHIFT = 8         # px the tape may move sideways between neighbouring rows
TRACE_GAP_ROWS = 6          # rows without the tape before the trace is considered ended
TRACE_WIDE_RATIO = 1.8      # a run this many times wider than the measured tape is a crossbar / branch row
BRANCH_REACH = 1.3          # a wide row must reach this many tape-widths beyond the tape centre to count as a branch
CONTINUE_ROWS = 8           # traced rows above the branch needed to say "the tape continues straight"
ANGLE_MIN_ROWS = 6          # v11: traced rows needed to measure the tape direction (few: short segments)
CROSS_ROW_FRACTION = 0.35   # v10 row-based junction search, kept as a fallback
CROSS_MIN_ROWS = 3
CROSS_SEARCH = (0.30, 0.95)

GREEN_LOWER = np.array([35, 70, 50], dtype=np.uint8)
GREEN_UPPER = np.array([90, 255, 255], dtype=np.uint8)
GREEN_MIN_AREA = 40
GREEN_MAX_AREA = 3000
GREEN_ROI_TOP = 0.45
GREEN_ROI_BOTTOM = 0.92
GREEN_TRIGGER_Y = 0.65
GREEN_SIDE_MARGIN = 8.0
GREEN_MAX_SIDE_DISTANCE = 65.0
GREEN_PAIR_Y_TOLERANCE = 0.15
MARKER_RAY_SCALE = 1.3      # v11: crossbar must lie within this many marker-sizes ahead of the marker

# corner = (direction, y): direction -1 left / +1 right, y as a fraction of image height; None if no corner
# line_angle: direction of the traced tape below any branch, degrees from straight ahead (+ = leans right), or None
# base: offset of the lowest traced tape rows (-1..1) - the tape directly in front of the robot, even below the near band
Observation = namedtuple("Observation", "positions direction junction_y zone_markers markers crossing corner line_angle base")

def detect_green(hsv, height):
    colour = cv2.inRange(hsv, GREEN_LOWER, GREEN_UPPER)
    kernel = np.ones((3, 3), dtype=np.uint8)
    colour = cv2.morphologyEx(colour, cv2.MORPH_OPEN, kernel)
    colour = cv2.morphologyEx(colour, cv2.MORPH_CLOSE, kernel)
    roi = colour.copy()
    roi[:int(height * GREEN_ROI_TOP), :] = 0
    roi[int(height * GREEN_ROI_BOTTOM):, :] = 0
    contours = cv2.findContours(roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    markers = []
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        if not GREEN_MIN_AREA <= area <= GREEN_MAX_AREA or h == 0 or not 0.4 <= w / float(h) <= 2.5:
            continue
        m = cv2.moments(contour)
        if m["m00"] <= 0:
            continue
        markers.append(dict(x=m["m10"] / m["m00"], y=m["m01"] / m["m00"], area=area, box=(x, y, w, h)))
    markers.sort(key=lambda item: item["area"], reverse=True)
    return markers, colour, roi

def row_runs(row):
    """Start/end columns of each continuous black run in one image row."""
    d = np.diff(np.concatenate(([0], row.astype(np.int8), [0])))
    return list(zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1) - 1))

def trace_line(mask):
    """v11: follow the robot's own tape upward, row by row, starting at the image bottom.

    A crossbar or corner shows up as a run much WIDER than the tape measured in the rows below it (a slanted
    crossbar is only ~2x the tape width, so the test is relative, not a fixed fraction of the image).
    Those rows are skipped for the position estimate and recorded as branches instead.
    Returns (xs: {row: column}, branch: {"LEFT": bool, "RIGHT": bool}, branch_y, continues).
    """
    height, width = mask.shape[:2]
    max_run = TRACE_MAX_RUN * width
    xs, branch, branch_y = {}, {"LEFT": False, "RIGHT": False}, None
    widths = deque(maxlen=8)
    x, miss = None, 0
    for y in range(int(TRACE_BOTTOM * height), int(TRACE_TOP * height) - 1, -1):
        runs = row_runs(mask[y] > 0)
        if x is None:   # acquire the tape nearest the centre, as low in the image as possible
            narrow = [r for r in runs if r[1] - r[0] + 1 <= max_run]
            if narrow:
                r = min(narrow, key=lambda r: abs((r[0] + r[1]) / 2.0 - width / 2.0))
                x = (r[0] + r[1]) / 2.0
                xs[y] = x
                widths.append(r[1] - r[0] + 1)
            continue
        touching = [r for r in runs if r[0] - TRACE_MAX_SHIFT <= x <= r[1] + TRACE_MAX_SHIFT]
        if not touching:
            miss += 1
            if miss > TRACE_GAP_ROWS:
                break
            continue
        miss = 0
        r = min(touching, key=lambda r: abs((r[0] + r[1]) / 2.0 - x))
        run_w = r[1] - r[0] + 1
        tape_w = float(np.median(widths)) if len(widths) >= 3 else None
        wide = run_w > max_run or (tape_w is not None and run_w > TRACE_WIDE_RATIO * tape_w and run_w > tape_w + 6)
        if not wide:
            x = (r[0] + r[1]) / 2.0
            xs[y] = x
            widths.append(run_w)
        else:           # crossbar / corner row: keep x, note which side(s) the stroke extends to
            reach = max(BRANCH_REACH * (tape_w or 10.0), 10.0)
            if x - r[0] > reach:
                branch["LEFT"] = True
            if r[1] - x > reach:
                branch["RIGHT"] = True
            if branch_y is None:
                branch_y = y
    continues = branch_y is not None and sum(1 for y in xs if y < branch_y) >= CONTINUE_ROWS
    return xs, branch, branch_y, continues

def find_junction_y(mask, height, width):
    """v10 fallback: lowest horizontal stroke as a fraction of image height, or None."""
    r0, r1 = int(CROSS_SEARCH[0] * height), int(CROSS_SEARCH[1] * height)
    counts = (mask[r0:r1] > 0).sum(axis=1)
    rows = np.flatnonzero(counts > CROSS_ROW_FRACTION * width)
    if rows.size == 0:
        return None
    runs = [r for r in np.split(rows, np.flatnonzero(np.diff(rows) > 1) + 1) if len(r) >= CROSS_MIN_ROWS]
    if not runs:
        return None
    return (r0 + float(runs[-1].mean())) / height

def detect_line(frame, gray, green_colour):
    height, width = frame.shape[:2]
    _, mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
    mask[cv2.dilate(green_colour, np.ones((5, 5), np.uint8)) > 0] = 0   # v10: green is never black tape
    overlay = frame.copy()
    xs, branch, branch_y, continues = trace_line(mask)
    positions, crossing = {}, set()
    for name, top, bottom in BANDS:
        y0, y1 = int(top * height), int(bottom * height)
        band = [x for y, x in xs.items() if y0 <= y < y1]
        positions[name] = None
        if len(band) >= 3:
            cx = float(np.mean(band))
            positions[name] = (cx - width / 2.0) / (width / 2.0)
            cv2.circle(overlay, (int(cx), (y0 + y1) // 2), 4, (0, 255, 0), -1)
            if branch_y is not None and y0 <= branch_y < y1:
                crossing.add(name)
        cv2.rectangle(overlay, (0, y0), (width - 1, y1), (255, 0, 0), 1)
    for y, x in xs.items():
        overlay[y, int(x)] = (255, 0, 255)
    # v11: tape direction from the traced rows BELOW any branch - works even when the segment is only a few cm long
    rows = sorted(y for y in xs if branch_y is None or y > branch_y)
    line_angle = None
    if len(rows) >= ANGLE_MIN_ROWS:
        slope = np.polyfit(np.array(rows, float), np.array([xs[y] for y in rows]), 1)[0]   # px sideways per row
        line_angle = float(np.degrees(np.arctan(-slope)))   # rows count upward = forward
    base = None
    if rows:
        low = rows[-6:]   # largest y = closest to the robot
        base = (float(np.mean([xs[y] for y in low])) - width / 2.0) / (width / 2.0)
    corner = None
    if branch_y is not None and not continues and branch["LEFT"] != branch["RIGHT"]:
        corner = (-1 if branch["LEFT"] else 1, branch_y / float(height))   # one-sided turn, tape ends: a corner
    return positions, crossing, mask, overlay, branch_y, corner, line_angle, base

def get_line_points(positions, width, height):
    band_y = {name: (top + bottom) / 2.0 for name, top, bottom in BANDS}
    return [(width * (1.0 + positions[name]) / 2.0, height * band_y[name])
            for name in ("far", "middle", "near") if positions.get(name) is not None]

def black_along(mask, marker, ux, uy, sign):
    """v11: first black pixel on a ray from the marker along the line direction (sign +1 ahead, -1 back)."""
    height, width = mask.shape[:2]
    _, _, w, h = marker["box"]
    size = max(w, h)
    start, stop = int(size / 2) + 3, int(size / 2 + MARKER_RAY_SCALE * size) + 4
    px_, py_ = -uy, ux
    for d in range(start, stop):
        for o in (-3, 0, 3):
            x = int(round(marker["x"] + sign * ux * d + o * px_))
            y = int(round(marker["y"] + sign * uy * d + o * py_))
            if 0 <= x < width and 0 <= y < height and mask[y, x]:
                return d, y
    return None

def classify_green_direction(positions, markers, width, height, mask):
    """Returns (direction, markers in trigger zone, estimated crossbar y as a fraction or None)."""
    zone = [m for m in markers if m["y"] >= height * GREEN_TRIGGER_Y]
    if not markers:
        return "NONE", 0, None
    points = get_line_points(positions, width, height)
    if len(points) < 2:
        return "UNKNOWN", len(zone), None
    near_x, near_y = points[-1]
    dx, dy = points[0][0] - near_x, points[0][1] - near_y
    length = (dx * dx + dy * dy) ** 0.5
    if length < 1e-6:
        return "UNKNOWN", len(zone), None
    dx, dy = dx / length, dy / length
    sides = {"LEFT": [], "RIGHT": []}
    crossbar_y = None
    for marker in zone:
        # v11: rule 1.5.6 checked ALONG THE LINE DIRECTION - works at any approach angle
        ahead = black_along(mask, marker, dx, dy, +1)
        behind = black_along(mask, marker, dx, dy, -1)
        if ahead is None or behind is not None:
            continue
        score = (marker["x"] - near_x) * dy - (marker["y"] - near_y) * dx
        if GREEN_SIDE_MARGIN < abs(score) <= GREEN_MAX_SIDE_DISTANCE:
            sides["LEFT" if score > 0 else "RIGHT"].append(marker)
            y = ahead[1] / float(height)
            crossbar_y = y if crossbar_y is None else max(crossbar_y, y)
    n = len(zone)
    if sides["LEFT"] and sides["RIGHT"]:
        paired = any(abs(a["y"] - b["y"]) <= height * GREEN_PAIR_Y_TOLERANCE
                     for a in sides["LEFT"] for b in sides["RIGHT"])
        return ("BOTH" if paired else "UNKNOWN"), n, crossbar_y
    if sides["LEFT"]:
        return "LEFT", n, crossbar_y
    if sides["RIGHT"]:
        return "RIGHT", n, crossbar_y
    return "CENTER", n, None

def perceive(frame):
    height, width = frame.shape[:2]
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    markers, green_colour, green_roi = detect_green(hsv, height)
    positions, crossing, mask, overlay, branch_y, corner, line_angle, base = detect_line(frame, gray, green_colour)
    junction_y = branch_y / float(height) if branch_y is not None else find_junction_y(mask, height, width)
    direction, zone, crossbar_y = classify_green_direction(positions, markers, width, height, mask)
    if junction_y is None and crossbar_y is not None:
        junction_y = crossbar_y
    return Observation(positions, direction, junction_y, zone, markers, crossing, corner, line_angle, base), overlay, green_roi

### 3b. Synthetic self-test (no motors)
Software-in-the-loop check of perception on drawn frames. Run it on the Jetson after changing any threshold.

In [ ]:
def synth(lines=(), markers=(), thickness=10):
    img = np.full((224, 224, 3), 235, np.uint8)
    for x0, y0, x1, y1 in lines:
        cv2.line(img, (x0, y0), (x1, y1), (20, 20, 20), thickness)
    for x, y in markers:
        cv2.rectangle(img, (x - 7, y - 7), (x + 7, y + 7), (40, 170, 40), -1)
    return img

def rotate_scene(img, degrees):
    """v11: rotate a synthetic scene about the bottom centre = robot approaching at an angle."""
    m = cv2.getRotationMatrix2D((112, 223), degrees, 1.0)
    return cv2.warpAffine(img, m, (224, 224), borderValue=(235, 235, 235))

def big(lines=(), markers=()):
    """Draw on a larger canvas so rotated scenes have no empty corners, then crop the camera view."""
    img = np.full((448, 448, 3), 235, np.uint8)
    for x0, y0, x1, y1 in lines:
        cv2.line(img, (x0 + 112, y0 + 224), (x1 + 112, y1 + 224), (20, 20, 20), 10)
    for x, y in markers:
        cv2.rectangle(img, (x + 105, y + 217), (x + 119, y + 231), (40, 170, 40), -1)
    return img

def diag(lines, markers, degrees):
    m = cv2.getRotationMatrix2D((224, 447), degrees, 1.0)
    return cv2.warpAffine(big(lines, markers), m, (448, 448), borderValue=(235, 235, 235))[224:448, 112:336]

def self_test(verbose=True):
    V = (112, 0, 112, 223)
    cases = [
        ("straight line", synth([V]), lambda o: abs(o.positions["middle"]) < 0.05 and o.junction_y is None),
        ("4-way: middle band not pulled sideways", synth([V, (0, 140, 223, 140)]),
         lambda o: abs(o.positions["middle"]) < 0.05 and o.junction_y is not None),
        ("T with left branch: keep through-line", synth([V, (0, 140, 112, 140)]),
         lambda o: abs(o.positions["middle"]) < 0.05),
        ("unmarked left corner -> corner signal LEFT", synth([(112, 223, 112, 140), (112, 140, 0, 140)]),
         lambda o: o.corner is not None and o.corner[0] == -1),
        ("unmarked right corner -> corner signal RIGHT", synth([(112, 223, 112, 140), (112, 140, 223, 140)]),
         lambda o: o.corner is not None and o.corner[0] == 1),
        ("T with branch, tape continues -> no corner", synth([V, (0, 140, 112, 140)]),
         lambda o: o.corner is None),
        ("zig-zag: first corner LEFT, not the far segment", synth([(112, 223, 112, 150), (112, 150, 70, 150), (70, 150, 70, 0)]),
         lambda o: o.corner is not None and o.corner[0] == -1 and abs(o.positions["near"]) < 0.05),
        ("diagonal 25 deg corner -> corner signal LEFT",
         diag([(112, -200, 112, 223)][:0] + [(112, 223, 112, 140), (112, 140, -200, 140)], [], 25),
         lambda o: o.corner is not None and o.corner[0] == -1),
        ("left marker before junction", synth([V, (0, 140, 223, 140)], [(94, 156)]),
         lambda o: o.direction == "LEFT"),
        ("right marker before junction", synth([V, (0, 140, 223, 140)], [(130, 156)]),
         lambda o: o.direction == "RIGHT"),
        ("dead end (both markers)", synth([V, (0, 140, 223, 140)], [(94, 156), (130, 156)]),
         lambda o: o.direction == "BOTH"),
        ("marker beyond the crossbar is ignored", synth([V, (0, 175, 223, 175)], [(94, 157)]),
         lambda o: o.direction not in ("LEFT", "RIGHT", "BOTH")),
        ("marker with no junction is ignored", synth([V], [(94, 170)]),
         lambda o: o.direction not in ("LEFT", "RIGHT", "BOTH")),
        # v11: approaching at an angle - the crossbar is not horizontal in the image
        ("diagonal 25 deg: left marker still LEFT",
         diag([(112, -200, 112, 223), (-200, 140, 423, 140)], [(94, 156)], 25), lambda o: o.direction == "LEFT"),
        ("diagonal -25 deg: right marker still RIGHT",
         diag([(112, -200, 112, 223), (-200, 140, 423, 140)], [(130, 156)], -25), lambda o: o.direction == "RIGHT"),
        ("diagonal 25 deg: both markers -> BOTH",
         diag([(112, -200, 112, 223), (-200, 140, 423, 140)], [(94, 156), (130, 156)], 25), lambda o: o.direction == "BOTH"),
        ("diagonal 25 deg: marker beyond crossbar ignored",
         diag([(112, -200, 112, 223), (-200, 175, 423, 175)], [(94, 157)], 25),
         lambda o: o.direction not in ("LEFT", "RIGHT", "BOTH")),
    ]
    passed = 0
    for name, img, check in cases:
        obs = perceive(img)[0]
        ok = bool(check(obs))
        passed += ok
        if verbose:
            pos = {k: (None if v is None else round(v, 2)) for k, v in obs.positions.items()}
            jy = None if obs.junction_y is None else round(obs.junction_y, 2)
            print("%-42s %s  dir=%-7s junction=%s %s" % (name, "PASS" if ok else "FAIL", obs.direction, jy, pos))
    print("%d/%d passed" % (passed, len(cases)))
    return passed == len(cases)

self_test()

## 4. BlackLineController
v11: planned 90° corners (approach → advance → rotate ≥ 63° → confirm alignment) driven by the perception corner signal; v10 reactive TURN kept for curves.

In [ ]:
TURN_ENTER = 0.55
TURN_EXIT = 0.22
TURN_EXIT_MIDDLE = 0.35        # v10: middle band need only be roughly aligned; the next bend may already be visible
LOOKAHEAD_GAIN = 0.40
ZIGZAG_SPREAD = 0.60           # v10: near and far on opposite sides by this much = two bends in view
TURN_PROGRESS = 0.03
TURN_STALL_TIMEOUT = 0.8       # v10: fail only if the error stops shrinking for this long
TURN_HARD_TIMEOUT = 0.5 * LINE_PIVOT_TIME_360   # v10: never rotate more than ~180 degrees
SEARCH_TIMEOUT = LINE_PIVOT_TIME_360 / 3.0      # v10: search up to ~120 degrees (v9: 0.35 s)

# v11: planned 90-degree corner, driven by the perception corner signal
# (tape bends to ONE side and does not continue). Same pattern as a green turn: approach -> advance -> rotate.
QUARTER_PIVOT_TIME = LINE_PIVOT_TIME_360 / 4.0
CORNER_CONFIRM_FRAMES = 2      # corner must be seen, same direction, in this many consecutive frames
CORNER_APPROACH_SPEED = MIN_DRIVE_SPEED
CORNER_APPROACH_TIMEOUT = 2.0
CORNER_NEAR_Y = 0.75           # like JUNCTION_NEAR_Y: corner must come this low before its disappearance counts
CORNER_EXIT_Y = 0.90
# v11: image bottom -> axle. CAMERA_BLIND_TIME was measured at BASE_SPEED, so scale it to the slower approach speed
CORNER_ADVANCE_TIME = CAMERA_BLIND_TIME * BASE_SPEED / CORNER_APPROACH_SPEED
CORNER_MIN_FRACTION = 0.70     # v11: rotate at least ~63 degrees before any tape may end the turn
CORNER_SLOW_RATIO = 0.70
CORNER_SLOW_ZONE = 0.60
CORNER_ALIGN_DEG = 7.0         # v11: traced tape within this angle of straight ahead = turn finished
                               # (perspective makes the measured angle about half the true one: 7 deg ~ 10-15 deg real)
CORNER_REACQUIRE_FRAMES = 2
CORNER_MAX_TIME = 2.0 * QUARTER_PIVOT_TIME
CORNER_SHORT_Y = 0.80          # v11: next corner this low = connector shorter than the visible tape; use rotation time
CORNER_CHAIN_Y = 0.60          # v11: chain into the next corner only if it is already this low in the image
CORNER_APPROACH_STEER = 0.08

class BlackLineController:
    """Follow black tape, handle sharp bends and search briefly when tape is lost.

    v11 modes: FOLLOW (proportional steering), CORNER_APPROACH / CORNER_ADVANCE / CORNER_TURN (planned
    90-degree corner, started by the perception corner signal), TURN (reactive pivot for curves the tracer
    still follows) and SEARCH (line lost).
    turn_started is set only while the robot is rotating, so the coordinator's green gating is unchanged.
    """
    def __init__(self):
        self.turn_direction = 0
        self.turn_started = None
        self.lost_started = None
        self.last_direction = 0
        self.stopped = False
        self.best_turn_error = None
        self.last_progress = None
        self.mode = "FOLLOW"           # v11
        self.mode_started = None
        self.corner_direction = 0
        self.aligned_frames = 0
        self.corner_seen = 0
        self.corner_last_y = 0.0
        self.corner_passed = 0
        self.pending_corner = None
        self.line_angle = None   # v11: set each frame by the coordinator from the perception tracer
        self.line_base = None

    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def pivot(self, direction, scale=1.0):
        speed = clamp(TURN_SPEED * scale, 0.0, MAX_SPEED)
        inner = clamp(PIVOT_INNER_RATIO * speed, -MAX_SPEED, 0.0)   # v10
        return (speed, inner) if direction > 0 else (inner, speed)

    def start_turn(self, direction, now, error=1.0):
        self.turn_started = now
        self.turn_direction = direction
        self.best_turn_error = abs(error)
        self.last_progress = now

    def enter(self, mode, now):
        self.mode = mode
        self.mode_started = now

    def corner_step(self, positions, now, corner=None):
        """v11: planned corner. Returns a command tuple."""
        near, middle = positions.get("near"), positions.get("middle")
        elapsed = now - self.mode_started

        if self.mode == "CORNER_APPROACH":
            if elapsed >= CORNER_APPROACH_TIMEOUT:
                return self.stop("corner approach timed out")
            if corner is not None:
                self.corner_last_y = corner[1]
                passed = corner[1] >= CORNER_EXIT_Y
            else:
                passed = self.corner_last_y >= CORNER_NEAR_Y
            self.corner_passed = self.corner_passed + 1 if passed else 0
            if self.corner_passed < 2:
                if near is None and middle is None and corner is None and self.corner_last_y < CORNER_NEAR_Y:
                    # v11: tape lost before reaching the corner - hand over to the bounded search, don't drive blind
                    self.enter("FOLLOW", now)
                    self.last_direction = self.corner_direction
                    return self.step(positions, now, corner)
                ref = [v for v in (near, middle) if v is not None]
                err = (0.6 * near + 0.4 * middle) if len(ref) == 2 else (ref[0] if ref else 0.0)
                steer = clamp(KP * err, -CORNER_APPROACH_STEER, CORNER_APPROACH_STEER)
                return (clamp(CORNER_APPROACH_SPEED + steer, 0.0, MAX_SPEED),
                        clamp(CORNER_APPROACH_SPEED - steer, 0.0, MAX_SPEED), "CORNER_APPROACH")
            self.enter("CORNER_ADVANCE", now)
            elapsed = 0.0

        if self.mode == "CORNER_ADVANCE":
            if elapsed < CORNER_ADVANCE_TIME:      # bring the rotation centre (axle) over the corner
                speed = clamp(CORNER_APPROACH_SPEED, 0.0, MAX_SPEED)
                return speed, speed, "CORNER_ADVANCE"
            self.enter("CORNER_TURN", now)
            self.start_turn(self.corner_direction, now)
            self.aligned_frames = 0
            self.pending_corner = None
            elapsed = 0.0

        # CORNER_TURN: open-loop for the first ~63 degrees (feedforward), then the camera closes the loop.
        if elapsed >= CORNER_MAX_TIME:
            return self.stop("corner turn did not find the outgoing line")
        if elapsed < CORNER_MIN_FRACTION * QUARTER_PIVOT_TIME:
            left, right = self.pivot(self.corner_direction)
            return left, right, "CORNER_TURN"
        if corner is not None:
            self.pending_corner = corner    # v11: next corner already in view (very short connector) - remember it
        # v11: finish only when the tape runs straight AHEAD: near band centred and the traced tape direction
        # within CORNER_ALIGN_DEG. Uses the tracer's measured angle, not the middle/far bands, so it also works when
        # the next corner is only a few cm away and those bands are empty. A tape crossing at a slant fails the angle.
        angle, base = self.line_angle, self.line_base
        centre = near if near is not None else base     # v11: next corner may fill the near band - use the lowest rows
        aligned = (centre is not None and abs(centre) < TURN_EXIT and angle is not None
                   and abs(angle) < CORNER_ALIGN_DEG)
        self.aligned_frames = self.aligned_frames + 1 if aligned else 0
        # v11: graceful degradation. If the NEXT corner is already at the bottom of the image, the connecting tape is
        # shorter than the camera can see (blind zone), so alignment cannot be confirmed visually. Finish on the
        # calibrated rotation time instead (feedforward), then chain into the next corner.
        very_short = (self.pending_corner is not None and self.pending_corner[1] >= CORNER_SHORT_Y
                      and elapsed >= QUARTER_PIVOT_TIME)
        if self.aligned_frames >= CORNER_REACQUIRE_FRAMES or very_short:
            self.turn_started = None
            self.turn_direction = 0
            pending, self.pending_corner = self.pending_corner, None
            if pending is not None and pending[1] >= CORNER_CHAIN_Y:   # chain only if the next corner is close
                # v11: chain straight into the next corner instead of losing it after the turn
                self.corner_direction = pending[0]
                self.last_direction = pending[0]
                self.corner_last_y, self.corner_passed = pending[1], 0
                self.enter("CORNER_APPROACH", now)
                return CORNER_APPROACH_SPEED, CORNER_APPROACH_SPEED, "CORNER_DONE_CHAIN"
            self.enter("FOLLOW", now)
            self.last_direction = 0
            return BASE_SPEED, BASE_SPEED, "CORNER_DONE"
        close = any(v is not None and abs(v) < CORNER_SLOW_ZONE for v in (near, middle))
        left, right = self.pivot(self.corner_direction, CORNER_SLOW_RATIO if close else 1.0)
        return left, right, "CORNER_TURN_SLOW" if close else "CORNER_TURN"

    def step(self, positions, now, corner=None):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        if self.mode != "FOLLOW":
            return self.corner_step(positions, now, corner)   # v11: a planned corner finishes before anything else
        # v11: debounce the corner signal, then start the planned maneuver instead of pivoting on the spot
        if corner is not None and (self.corner_seen == 0 or corner[0] == self.corner_direction):
            self.corner_direction = corner[0]
            self.corner_seen += 1
        else:
            self.corner_seen = 0
        if self.corner_seen >= CORNER_CONFIRM_FRAMES and self.turn_started is None:
            self.corner_seen = 0
            self.corner_last_y, self.corner_passed = corner[1], 0
            self.enter("CORNER_APPROACH", now)
            return self.corner_step(positions, now, corner)
        near, middle = positions.get("near"), positions.get("middle")
        visible = [(name, value) for name, value in positions.items() if value is not None]

        if not visible:
            if self.lost_started is None:
                self.lost_started = now
            if not self.last_direction:
                return self.stop("line lost without a known turn direction")
            if now - self.lost_started >= SEARCH_TIMEOUT:
                return self.stop("line not reacquired")
            if self.turn_started is None:
                self.start_turn(self.last_direction, now)
            if now - self.turn_started >= TURN_HARD_TIMEOUT:
                return self.stop("turn timed out")
            left, right = self.pivot(self.turn_direction)
            return left, right, "SEARCH"

        returning = self.lost_started is not None
        self.lost_started = None
        weights = {"near": 0.45, "middle": 0.35, "far": 0.20}
        far = positions.get("far")
        farthest = far if far is not None else middle
        zigzag = (near is not None and farthest is not None and near * farthest < 0
                  and abs(farthest - near) >= ZIGZAG_SPREAD)
        if zigzag:
            used = [(n, v) for n, v in visible if n != "far"]
            error = sum(weights[n] * v for n, v in used) / sum(weights[n] for n, _ in used)
        else:
            error = sum(weights[n] * v for n, v in visible) / sum(weights[n] for n, _ in visible)
            if near is not None and farthest is not None:
                error += LOOKAHEAD_GAIN * (farthest - near)
        error = clamp(error, -1.0, 1.0)
        if abs(error) > 0.15:
            self.last_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            if returning:
                self.best_turn_error, self.last_progress = abs(error), now
            if near is not None and abs(near) < TURN_EXIT and (middle is None or abs(middle) < TURN_EXIT_MIDDLE):
                self.turn_started = None
                self.turn_direction = 0
            else:
                if abs(error) < self.best_turn_error - TURN_PROGRESS:
                    self.best_turn_error, self.last_progress = abs(error), now
                if now - self.last_progress >= TURN_STALL_TIMEOUT:
                    return self.stop("turn made no progress")
                if now - self.turn_started >= TURN_HARD_TIMEOUT:
                    return self.stop("turn timed out")

        if self.turn_started is None and abs(error) >= TURN_ENTER:
            self.start_turn(1 if error > 0 else -1, now, error)

        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        bend = min(abs(error) / TURN_ENTER, 1.0)
        base = BASE_SPEED + (CURVE_SPEED - BASE_SPEED) * bend
        base = MIN_DRIVE_SPEED if zigzag else max(base, MIN_DRIVE_SPEED)
        correction = KP * error
        left = clamp(base + correction, 0.0, MAX_SPEED)
        right = clamp(base - correction, 0.0, MAX_SPEED)
        return left, right, ("FOLLOW_SLOW" if zigzag else "FOLLOW")

## 5. GapCrossingController (unchanged from v9)

In [ ]:
GAP_SPEED = BASE_SPEED
GAP_TIMEOUT = 1.5
GAP_STEER_LIMIT = 0.015
GAP_APPROACH_FRAMES = 3
GAP_APPROACH_MAX_AGE = 0.6
GAP_ALIGNMENT = 0.25
GAP_STRAIGHT_SPREAD = 0.12
GAP_CONFIRM_FRAMES = 2

class GapCrossingController:
    """Own gap evidence/crossing; delegate ordinary driving to BlackLineController."""
    def __init__(self):
        self.black_line = BlackLineController()
        self.straight_frames = 0
        self.straight_seen = None
        self.saved_correction = 0.0
        self.gap_started = None
        self.confirm_frames = 0

    def gap_commands(self):
        return (clamp(GAP_SPEED + self.saved_correction, 0.0, MAX_SPEED),
                clamp(GAP_SPEED - self.saved_correction, 0.0, MAX_SPEED),
                "GAP: forward, awaiting tape")

    def step(self, positions, now, corner=None):
        if self.black_line.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near, middle = positions.get("near"), positions.get("middle")
        values = [v for v in positions.values() if v is not None]

        if self.gap_started is not None:
            if now - self.gap_started >= GAP_TIMEOUT:
                return self.black_line.stop("gap crossing timed out")
            self.confirm_frames = self.confirm_frames + 1 if (near is not None and middle is not None) else 0
            if self.confirm_frames < GAP_CONFIRM_FRAMES:
                return self.gap_commands()
            self.gap_started = None
            self.confirm_frames = 0
            self.black_line.lost_started = None
            self.black_line.last_direction = 0

        if not values:
            recent = self.straight_seen is not None and now - self.straight_seen <= GAP_APPROACH_MAX_AGE
            # v11: never start a gap crossing in the middle of a planned corner maneuver
            if (self.black_line.turn_started is None and self.black_line.mode == "FOLLOW" and recent
                    and self.straight_frames >= GAP_APPROACH_FRAMES):
                self.gap_started = now
                self.straight_frames = 0
                self.straight_seen = None
                self.confirm_frames = 0
                return self.gap_commands()
            return self.black_line.step(positions, now, corner)

        left, right, state = self.black_line.step(positions, now, corner)
        aligned = max(abs(v) for v in values) <= GAP_ALIGNMENT
        straight = len(values) >= 2 and max(values) - min(values) <= GAP_STRAIGHT_SPREAD
        if state == "FOLLOW" and near is not None and aligned and straight:
            if self.straight_seen is None or now - self.straight_seen > GAP_APPROACH_MAX_AGE:
                self.straight_frames = 0
            self.straight_frames += 1
            self.straight_seen = now
            correction = clamp((left - right) / 2.0, -GAP_STEER_LIMIT, GAP_STEER_LIMIT)
            self.saved_correction = 0.5 * self.saved_correction + 0.5 * correction
        elif state != "FOLLOW" or not aligned or (len(values) >= 2 and not straight):
            self.straight_frames = 0
            self.straight_seen = None
        return left, right, state

## 6. GreenMarkerTurnController
v10 sequence: vote → STOP (may upgrade to BOTH) → ADVANCE until the crossbar passes under the camera,
then CAMERA_BLIND_TIME → TURN (camera confirms reacquisition) → DONE. A dead end (BOTH) skips the advance.

In [14]:
GREEN_VOTE_WINDOW = 5          # v10: k-of-n confirmation
GREEN_VOTES_NEEDED = 3
GREEN_BOTH_UPGRADE = 2         # v10: BOTH frames seen while stopped that turn a LEFT/RIGHT into a U-turn
GREEN_STOP_TIME = 0.20
GREEN_ADVANCE_SPEED = BASE_SPEED
GREEN_ADVANCE_STEER = 0.05
JUNCTION_NEAR_Y = 0.75         # crossbar must have come this low before its disappearance counts
JUNCTION_EXIT_Y = 0.90         # ...or reached this row
JUNCTION_JUMP = 0.20           # a stroke this much higher is a different (farther) junction
GREEN_NO_JUNCTION_WAIT = 0.30  # crossbar never seen while advancing: assume already under the camera
GREEN_ADVANCE_TIMEOUT = 2.0
GREEN_TURN_MIN_TIME = 0.6 * QUARTER_TURN_TIME
GREEN_TURN_TIMEOUT = 2.5 * QUARTER_TURN_TIME
UTURN_MIN_TIME = 0.75 * 2 * QUARTER_TURN_TIME
UTURN_TIMEOUT = 2.0 * 2 * QUARTER_TURN_TIME
GREEN_REACQUIRE_FRAMES = 3
GREEN_COOLDOWN = 0.4
GREEN_CLEAR_FRAMES = 0         # v10: 0 = cooldown only; the junction-above rule already blocks the old marker
GREEN_TURN_SLOW_SPEED = 0.10   # CALIBRATE: lowest speed that still reliably rotates in place
GREEN_SLOW_ZONE = 0.6          # line within this offset in near or middle -> slow down
FEEDFORWARD_FRACTION = 0.75    # fraction of the expected angle done fast, open-loop

def line_centered(positions):
    return all(positions.get(n) is not None and abs(positions[n]) < TURN_EXIT for n in ("near", "middle"))

class GreenMarkerTurnController:
    """Confirm green instructions and execute them; return None when not in control."""
    def __init__(self):
        self.state = "FOLLOW"
        self.state_started = None
        self.direction = None
        self.votes = deque(maxlen=GREEN_VOTE_WINDOW)
        self.both_votes = 0
        self.armed = True
        self.clear_frames = 0
        self.cooldown_until = 0.0
        self.departed = False
        self.reacquire_frames = 0
        self.stopped = False
        self.reason = ""

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        return 0.0, 0.0, "STOP: " + reason

    def enter(self, state, now):
        self.state = state
        self.state_started = now

    def turn(self, positions, elapsed):
        expected = (2 if self.direction == "BOTH" else 1) * QUARTER_TURN_TIME
        near, middle = positions.get("near"), positions.get("middle")
        line_close = any(v is not None and abs(v) < GREEN_SLOW_ZONE for v in (near, middle))
        slow = elapsed >= FEEDFORWARD_FRACTION * expected and line_close
        speed = clamp(GREEN_TURN_SLOW_SPEED if slow else GREEN_TURN_SPEED, 0.0, MAX_SPEED)
        suffix = "_SLOW" if slow else ""
        if self.direction == "RIGHT":
            return speed, -speed, "GREEN_TURN_RIGHT" + suffix
        label = "GREEN_U_TURN" if self.direction == "BOTH" else "GREEN_TURN_LEFT"
        return -speed, speed, label + suffix

    def confirmed(self):
        counts = {d: self.votes.count(d) for d in ("LEFT", "RIGHT", "BOTH")}
        best = max(counts, key=counts.get)
        return best if counts[best] >= GREEN_VOTES_NEEDED else None

    def begin_advance(self, now):
        self.junction_seen = False
        self.last_junction_y = None
        self.passed_frames = 0
        self.post_started = None
        self.enter("GREEN_ADVANCE", now)

    def begin_turn(self, now):
        self.departed = False
        self.reacquire_frames = 0
        self.enter("GREEN_TURN", now)

    def advance(self, obs, now):
        """v10: closed-loop approach. Returns a command, or None when the axle is over the junction."""
        elapsed = now - self.state_started
        if elapsed >= GREEN_ADVANCE_TIMEOUT:
            return self.stop("junction not passed during advance")
        if self.post_started is None:
            jy = obs.junction_y
            if jy is not None and (self.last_junction_y is None or jy >= self.last_junction_y - JUNCTION_JUMP):
                self.junction_seen = True
                self.last_junction_y = jy
                under = jy >= JUNCTION_EXIT_Y
            elif self.junction_seen:
                under = self.last_junction_y >= JUNCTION_NEAR_Y
            else:
                under = elapsed >= GREEN_NO_JUNCTION_WAIT
            self.passed_frames = self.passed_frames + 1 if under else 0
            if self.passed_frames >= 2:
                self.post_started = now
        elif now - self.post_started >= CAMERA_BLIND_TIME:
            return None
        near = obs.positions.get("near")
        steer = clamp(KP * near, -GREEN_ADVANCE_STEER, GREEN_ADVANCE_STEER) if near is not None else 0.0
        speed = clamp(GREEN_ADVANCE_SPEED, 0.0, MAX_SPEED)
        label = "GREEN_ADVANCE_BLIND" if self.post_started is not None else "GREEN_ADVANCE"
        return clamp(speed + steer, 0.0, MAX_SPEED), clamp(speed - steer, 0.0, MAX_SPEED), label

    def step(self, obs, now, can_start=True):
        if self.stopped:
            return self.stop(self.reason)
        positions, direction = obs.positions, obs.direction

        if self.state == "GREEN_STOP":
            if direction == "BOTH":
                self.both_votes += 1
            if self.direction != "BOTH" and self.both_votes >= GREEN_BOTH_UPGRADE:
                self.direction = "BOTH"     # v10: late binding - second marker came into view
            if now - self.state_started < GREEN_STOP_TIME:
                return 0.0, 0.0, "GREEN_STOP"
            if self.direction == "BOTH":
                self.begin_turn(now)        # v10: dead end - turn around before entering the junction
            else:
                self.begin_advance(now)

        if self.state == "GREEN_ADVANCE":
            command = self.advance(obs, now)
            if command is not None:
                return command
            self.begin_turn(now)

        if self.state == "GREEN_TURN":
            elapsed = now - self.state_started
            limit = UTURN_TIMEOUT if self.direction == "BOTH" else GREEN_TURN_TIMEOUT
            minimum = UTURN_MIN_TIME if self.direction == "BOTH" else GREEN_TURN_MIN_TIME
            if elapsed >= limit:
                return self.stop("green turn timed out")
            centered = line_centered(positions)
            if not centered:
                self.departed = True
            if self.departed and elapsed >= minimum and centered:
                self.reacquire_frames += 1
            else:
                self.reacquire_frames = 0
            if self.reacquire_frames < GREEN_REACQUIRE_FRAMES:
                return self.turn(positions, elapsed)
            self.enter("FOLLOW", now)
            self.cooldown_until = now + GREEN_COOLDOWN
            self.clear_frames = 0
            self.votes.clear()
            return 0.0, 0.0, "GREEN_DONE"

        if not self.armed:
            self.clear_frames = self.clear_frames + 1 if direction not in ("LEFT", "RIGHT", "BOTH") else 0
            if now >= self.cooldown_until and self.clear_frames >= GREEN_CLEAR_FRAMES:
                self.armed = True

        if self.armed and can_start:
            self.votes.append(direction)
            decided = self.confirmed()
            if decided is not None:
                self.direction = decided
                self.armed = False
                self.votes.clear()
                self.both_votes = 0
                self.enter("GREEN_STOP", now)
                return 0.0, 0.0, "GREEN_STOP"
        else:
            self.votes.clear()
        return None

## 7. NavigationCoordinator
v10: takes an `Observation`. A green maneuver may now start during a black-line TURN **if a junction is
visible** (that "turn" was caused by the crossbar). It still never interrupts a gap crossing.

In [ ]:
class NavigationCoordinator:
    """Single decision point: green maneuver > gap crossing > black-line control."""
    def __init__(self):
        self.green_turn = GreenMarkerTurnController()
        self.gap = GapCrossingController()
        self.stopped = False
        self.reason = ""
        self.active_controller = "BLACK_LINE"

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        self.active_controller = "STOPPED"
        return 0.0, 0.0, "STOP: " + reason

    def step(self, obs, now):
        if self.stopped:
            return self.stop(self.reason)
        line_turning = self.gap.black_line.turn_started is not None
        can_start_green = (self.gap.gap_started is None and
                           (not line_turning or obs.junction_y is not None))   # v10
        command = self.green_turn.step(obs, now, can_start=can_start_green)
        if command is not None:
            self.active_controller = "GREEN_TURN"
            if self.green_turn.stopped:
                self.stopped = True
                self.reason = self.green_turn.reason
                self.active_controller = "STOPPED"
            elif command[2] == "GREEN_DONE":
                self.gap = GapCrossingController()   # discard pre-turn history
            return command
        self.gap.black_line.line_angle = obs.line_angle   # v11: measured tape direction for corner completion
        self.gap.black_line.line_base = obs.base
        command = self.gap.step(obs.positions, now, obs.corner)   # v11: corner signal for planned turns
        self.active_controller = "GAP_CROSSING" if self.gap.gap_started is not None else "BLACK_LINE"
        if self.gap.black_line.stopped:
            return self.stop(command[2])
        return command

## 8. Runner + logging
The control loop does no file I/O: each frame appends one tuple to a bounded ring buffer, and the CSV is
written after the motors have stopped. The summary reports loop rate, processing time and frame age -
use these numbers in the report's timing discussion.

In [ ]:
LOG_FIELDS = ("t", "frame_age", "proc_ms", "owner", "state", "marker", "near", "middle", "far",
              "junction_y", "zone_markers", "left", "right", "corner")
LOG = deque(maxlen=6000)   # ~5 min at 20 Hz; memory stays bounded

def summarize_log(save=True):
    rows = list(LOG)
    if len(rows) < 2:
        print("No log data.")
        return
    t = np.array([r[0] for r in rows]); proc = np.array([r[2] for r in rows]); age = np.array([r[1] for r in rows])
    periods = np.diff(t)
    print("frames %d | loop %.1f Hz (period mean %.0f ms, max %.0f ms) | processing mean %.1f ms, p95 %.1f ms, max %.1f ms | frame age max %.0f ms"
          % (len(rows), 1.0 / periods.mean(), 1000 * periods.mean(), 1000 * periods.max(),
             proc.mean(), np.percentile(proc, 95), proc.max(), 1000 * age.max()))
    previous = None
    for r in rows:
        key = (r[3], r[4].split(":")[0])
        if key != previous:
            print("  t=%6.2f  %-12s %-22s marker=%-7s junction=%s" % (r[0], r[3], r[4], r[5],
                  "-" if r[9] is None else "%.2f" % r[9]))
            previous = key
    if save:
        name = "run_%d.csv" % int(time.time())
        with open(name, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(LOG_FIELDS)
            writer.writerows(rows)
        print("Saved", name)

In [ ]:
overlay_view = widgets.Image(format="jpeg", width=336, height=336)
mask_view = widgets.Image(format="jpeg", width=336, height=336)
status_view = widgets.Label(value="Ready for stationary preview")
display(widgets.VBox([widgets.HBox([overlay_view, mask_view]), status_view]))

def run_v10(seconds=10, drive=False, show=True):
    robot.stop()
    controller = NavigationCoordinator()
    LOG.clear()
    start = time.monotonic()
    deadline = start + seconds
    previous_stamp = None
    next_display = 0.0
    display_period = 0.3 if drive else 0.1   # v10: JPEG encoding is non-essential work; do less of it while driving
    try:
        while time.monotonic() < deadline:
            sample = latest["sample"]
            if sample is None:
                print("Stopped: no camera frame")
                break
            frame, captured_at = sample
            now = time.monotonic()
            if now - captured_at > FRAME_TIMEOUT:
                print("Stopped: stale camera frame")
                break
            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue
            previous_stamp = captured_at
            t0 = time.monotonic()
            obs, overlay, green_mask = perceive(frame)
            now = time.monotonic()
            proc_ms = 1000.0 * (now - t0)
            if now >= deadline or now - captured_at > FRAME_TIMEOUT:
                print("Stopped: deadline or slow image processing")
                break
            left, right, state = controller.step(obs, now)
            if drive:
                robot.set_motors(left, right)
            p = obs.positions
            LOG.append((now - start, now - captured_at, proc_ms, controller.active_controller, state,
                        obs.direction, p.get("near"), p.get("middle"), p.get("far"),
                        obs.junction_y, obs.zone_markers, left, right,
                        None if obs.corner is None else "%+d@%.2f" % obs.corner))   # v11
            if show and now >= next_display:
                height, width = frame.shape[:2]
                points = get_line_points(p, width, height)
                for a, b in zip(points, points[1:]):
                    cv2.line(overlay, tuple(map(int, a)), tuple(map(int, b)), (255, 0, 255), 2)
                for marker in obs.markers:
                    x, y, w, h = marker["box"]
                    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)
                if obs.junction_y is not None:
                    jy = int(obs.junction_y * height)
                    cv2.line(overlay, (0, jy), (width - 1, jy), (0, 0, 255), 1)
                overlay_view.value = bgr8_to_jpeg(overlay)
                mask_view.value = bgr8_to_jpeg(cv2.cvtColor(green_mask, cv2.COLOR_GRAY2BGR))
                status_view.value = "%s | %s | marker=%s | L=%.2f R=%.2f | owner=%s" % (
                    "DRIVE" if drive else "PREVIEW", state, obs.direction, left, right, controller.active_controller)
                next_display = now + display_period
            if controller.stopped:
                print(state)
                break
            time.sleep(0.005)
    finally:
        robot.stop()
        print("Run ended; motors stopped.")
        summarize_log()

def run_preview(seconds=20):
    run_v10(seconds=seconds, drive=False)

def run_robot(seconds=2):
    run_v10(seconds=seconds, drive=True)

## 9. Test order
1. `self_test()` (cell 3b). 2. `spin_test()` → set `ROTATION_TIME_360`, re-run cell 2 and the controller cells.
3. Measure `CAMERA_BLIND_TIME`: tape across the floor, drive straight, note when it leaves the image and when it is under the axle.
4. `run_preview()` pushing the robot by hand over a marked junction: watch for the red junction line and `marker=LEFT`.
5. Floor: single left / right / both marker, then the back-to-back marker section, then tight bends and gaps.
After every run read the printed transition list — it shows which state ended the run.

In [27]:
run_preview(seconds=30)

STOP: STOP: turn made no progress
Run ended; motors stopped.
frames 88 | loop 30.1 Hz (period mean 33 ms, max 39 ms) | processing mean 3.7 ms, p95 5.0 ms, max 5.5 ms | frame age max 13 ms
  t=  0.01  BLACK_LINE   FOLLOW                 marker=NONE    junction=-
  t=  1.56  BLACK_LINE   TURN                   marker=NONE    junction=0.58
  t=  2.90  STOPPED      STOP: STOP: turn made no progress marker=NONE    junction=-
Saved run_1792934466.csv


In [30]:
# Explicitly uncomment to move the robot:
run_robot(seconds=20)

Run ended; motors stopped.
frames 589 | loop 29.4 Hz (period mean 34 ms, max 96 ms) | processing mean 3.7 ms, p95 4.8 ms, max 9.7 ms | frame age max 44 ms
  t=  0.01  BLACK_LINE   FOLLOW                 marker=NONE    junction=-
  t=  0.67  BLACK_LINE   FOLLOW_SLOW            marker=NONE    junction=0.55
  t=  0.88  BLACK_LINE   TURN                   marker=NONE    junction=0.81
  t=  1.34  BLACK_LINE   FOLLOW                 marker=NONE    junction=-
  t=  1.57  BLACK_LINE   FOLLOW_SLOW            marker=NONE    junction=-
  t=  1.77  BLACK_LINE   TURN                   marker=NONE    junction=-
  t=  2.10  BLACK_LINE   FOLLOW                 marker=NONE    junction=-
  t=  2.41  BLACK_LINE   FOLLOW_SLOW            marker=CENTER  junction=-
  t=  2.44  BLACK_LINE   TURN                   marker=CENTER  junction=0.52
  t=  2.64  GREEN_TURN   GREEN_STOP             marker=LEFT    junction=0.63
  t=  2.84  GREEN_TURN   GREEN_ADVANCE          marker=LEFT    junction=0.75
  t=  3.17  GREE

## 10. Cleanup

In [ ]:
robot.stop()
camera.unobserve(receive_frame, names="value")
camera_link.unlink()
camera.stop()